#  From Business Problem to ML Model: A Fraud Detection Walkthrough
### Session 1 of 2

---

Welcome! This notebook is your **teaching guide, code reference, and practice material** all in one. Work through it cell by cell, read the explanations, run the code, and attempt the exercises.

**What we will cover today:**
- Framing a real-world fraud detection problem
- Exploring and understanding the data
- Preprocessing for ML
- The accuracy trap (why accuracy lies on imbalanced data)
- Handling imbalance at the algorithm level
- Evaluating model performance properly
- Saving the model for deployment

**Session 2 Preview:**  
In the next session, we will take the model we build today and **deploy it as a REST API using FastAPI**, so it can serve real-time fraud predictions. Make sure you keep this notebook and the saved model file!

---

>  **Tip:** If you are new to ML, do not worry about memorising everything. Focus on understanding the *reasoning* behind each step.

## 1. Problem Framing

Before writing a single line of code, a good ML practitioner asks: **what is the business problem?**

### The Scenario
A financial institution processes hundreds of thousands of credit card transactions daily. A small fraction of these are fraudulent. The company wants an automated system that can **flag suspicious transactions** so their fraud team can investigate.

### Translating to an ML Problem

| Business Question | ML Translation |
|---|---|
| Is this transaction fraud? | Binary classification (fraud = 1, not fraud = 0) |
| How confident are we? | Predict a probability score |
| What counts as a mistake? | Missing fraud (false negative) is costly; false alarms waste analyst time |

### Key Insight: Not All Errors Are Equal
In fraud detection:
- **Missing a fraud** (False Negative) → customer loses money, bank loses trust
- **Flagging a legitimate transaction** (False Positive) → customer is inconvenienced

This means we care deeply about **recall** (catching as much fraud as possible), while keeping **precision** reasonable. We will come back to this during evaluation.

---

## 2.  Setup & Data Loading

### Dataset: Credit Card Fraud Detection (Kaggle)

We are using the [Credit Card Fraud Detection dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) from Kaggle.  
It contains **284,807 transactions** made by European cardholders over two days in September 2013.

**Features:**
- `V1` to `V28` — PCA-transformed features (anonymised for privacy)
- `Time` — seconds elapsed between this transaction and the first transaction
- `Amount` — transaction amount
- `Class` — target variable (1 = fraud, 0 = legitimate)

In [ ]:
# Core data handling
import pandas as pd        # For working with tabular data
import numpy as np         # For numerical operations

# Visualization
import matplotlib.pyplot as plt   # Basic plotting
import seaborn as sns             # Better-looking plots

# Model building and preprocessing
from sklearn.model_selection import train_test_split   # Split data into train/test
from sklearn.preprocessing import StandardScaler       # Scale features
from sklearn.linear_model import LogisticRegression    # Classification model

# Evaluation metrics
from sklearn.metrics import (
    confusion_matrix,        # Shows prediction breakdown
    classification_report,    # Precision, Recall, F1-score
    accuracy_score          # Overall correctness of predictions
)


# Utility
import joblib                 # For saving/loading models (used later in deployment)
import warnings
warnings.filterwarnings('ignore')  # Hide unnecessary warnings for cleaner output

# Set consistent plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100   # Improve plot clarity

In [ ]:
# Load the dataset

url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
df = pd.read_csv(url)

# Display the first 7 rows
df.head(7)

# Question:
# How do I display the first 5 rows


In [ ]:
df.head()

## 3.  Exploratory Data Analysis (EDA)

Before modelling, we need to **understand the data**. Good EDA helps in making better modelling decisions.
- Which features (columns) should I use in building my model and which should I not use. (Time of transaction, location etc)

In [ ]:
# Check the Shape of the data
df.shape

# Question:
# What do the numbers mean?

In [ ]:
# Check basic information
df.info()

# Are there missing values? What are the datatypes of the columns

In [ ]:
# Calculate the total missing values
df.isnull().sum().sum()

#df.isnull().sum()

In [ ]:
# Check Class Distribution
# Check how many fraud vs non-fraud cases

df['Class'].value_counts()

# Question
# Is it balanced?
#

In [ ]:
# Convert distribution to percentages

df['Class'].value_counts(normalize=True)

In [ ]:
# Let's visualize the  Class distribution
class_counts = df['Class'].value_counts()

print(' Class Distribution ')


# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], class_counts.values,
            color=['steelblue', 'tomato'], edgecolor='white', linewidth=0.8)
axes[0].set_title('Transaction Class Counts', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

# Pie chart
axes[1].pie(class_counts.values, labels=['Legitimate', 'Fraud'],
            autopct='%1.3f%%', colors=['steelblue', 'tomato'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Class Balance', fontweight='bold')

plt.suptitle(' Highly Imbalanced Dataset', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Group transactions by class (0 = normal, 1 = fraud)
# Then analyze how the transaction amounts differ
(
df.groupby('Class')['Amount']   # Group data by fraud (1) and non-fraud (0), then select Amount
.describe()                   # Get summary stats (mean, min, max, etc.)
  .round(2)                     # Round values for cleaner display
  )

## 4. Preprocessing

Our data is fairly clean (no missing values). We need to:
1. **Split into train and test sets**
2. **Scale the data** (Technically only Amount and Time need scaling, but for simplicity and consistency, we scale all features, especially since Logistic Regression is sensitive to feature scale)

In [ ]:
# Split into X and y

X = df.drop('Class', axis=1)
y = df['Class']

In [ ]:
# Perform train-test split

# Split data into training (80%) and testing (20%)
# stratify=y ensures the class distribution is preserved
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Show how the "Stratification" preserved proper distribution

print(f'Training set:   {X_train.shape[0]:,} samples')
print(f'Test set:       {X_test.shape[0]:,} samples')
print(f'\nFraud in train: {y_train.sum():,} ({y_train.mean()*100:.3f}%)')
print(f'Fraud in test:  {y_test.sum():,}  ({y_test.mean()*100:.3f}%)')


In [ ]:
# Scale the data

# We use StandardScaler to standardize the features
# This transforms the data so that each feature has:
# - mean = 0
# - standard deviation = 1

scaler = StandardScaler()

# Fit the scaler on the training data and transform it
# IMPORTANT:
# We only "fit" on training data so the model does not learn from test data

X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler (already fitted on training data) to transform the test data
# This ensures both train and test data are on the same scale

X_test_scaled = scaler.transform(X_test)

Compare

In [ ]:
X_train.head()

In [ ]:
X_train_scaled[:4]

## 5. Baseline Model

In [ ]:
# Train a Logistic Regression model

# Initialize the model
# max_iter=1000 ensures the model has enough iterations to converge
model = LogisticRegression(max_iter=1000)

# Fit the model on the training data
# The model learns the relationship between features (X) and target (y)
model.fit(X_train_scaled, y_train)

In [ ]:
# Predict on the test set

# Use the trained model to make predictions on unseen data
# This helps us evaluate how well the model generalizes
y_pred = model.predict(X_test_scaled)

## 6.  The Accuracy Trap

This is one of the most important lessons in ML with imbalanced data.  

In [ ]:
# Evaluate model using accuracy

# Accuracy = (correct predictions) / (total predictions)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

At first glance, this accuracy may look super good.

But remember:
- Fraud cases are very rare  
- A model can predict "Not Fraud" most of the time and still achieve high accuracy  

 So accuracy alone can be misleading in imbalanced datasets

### Reality Check

In [ ]:
fraud_caught = y_pred[y_test == 1].sum()
total_fraud  = y_test.sum()

print(' Naive Model (no imbalance handling) ')
print(f'Accuracy:          {accuracy:.4f}  ← looks great, right? ')
print(f'Fraud caught:      {fraud_caught} out of {total_fraud}')
print(f'Fraud missed:      {total_fraud - fraud_caught}  ← this is the real story')

In [ ]:
# Generate confusion matrix

# This shows how predictions compare to actual values
# Format:
# [[TN, FP],
#  [FN, TP]]
cm = confusion_matrix(y_test, y_pred)

# Visualize confusion matrix

# Add labels for better interpretation
sns.heatmap(
    cm,
    annot=True,          # Show numbers inside cells
    fmt='d',             # Format as integers
    cmap='Blues',
    xticklabels=['Legitimate', 'Fraud'],   # Predicted labels
    yticklabels=['Legitimate', 'Fraud']    # Actual labels
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

 **Questions:**

- How many fraud cases were correctly detected?  
- How many were missed?  

###  Understanding the Confusion Matrix

```
                    Predicted Legitimate    Predicted Fraud
Actual Legitimate        True Negative (TN)    False Positive (FP)
Actual Fraud             False Negative (FN)   True Positive (TP)
```

| Term | Meaning in Fraud Context |
|------|-------------------------|
| **True Positive (TP)** | Fraud correctly flagged ✅ |
| **True Negative (TN)** | Legitimate correctly cleared ✅ |
| **False Positive (FP)** | Legitimate flagged as fraud ❌ (customer frustrated) |
| **False Negative (FN)** | Fraud missed ❌ (most costly!) |

> **Key takeaway:** High accuracy means nothing if the model is simply predicting "not fraud" for almost every transaction. We need better metrics.

---

In [ ]:
# Detailed evaluation metrics

print(classification_report(y_test, y_pred))

## Key Insight

Focus on:

- **Recall (Fraud class)** → How many fraud cases we correctly detect  
- **Precision** → How many predicted fraud cases are actually fraud  

 In fraud detection, recall is especially important

 | Metric | Formula | What it asks |
|--------|---------|-------------|
| **Precision** | TP / (TP + FP) | Of all flagged transactions, how many were actually fraud? |
| **Recall** | TP / (TP + FN) | Of all actual fraud, how many did we catch? |
| **F1-Score** | 2 × (P × R) / (P + R) | Harmonic mean of precision and recall |

## 7. HANDLING IMBALANCED DATA


We saw that our model struggles to detect fraud.

There are different ways to handle imbalanced data.

In this session, we’ll focus on a simple and effective approach:

 **Change how the model learns**

We do this using **class weights**, which makes the model pay more attention to the fraud class.


## Handling Imbalanced Data with Class Weights

The fix is simple:

 Tell the model to **penalise mistakes on the minority class more heavily**. *(care more about the minority class (fraud))*

In scikit-learn, this is as easy as:

`class_weight='balanced'`

### What does this mean?

Instead of treating all mistakes equally, the model:
- Penalises mistakes on fraud cases more heavily  
- Pays more attention to the minority class during training  

 So missing a fraud case becomes “more costly” to the model

---

### How does it work under the hood?

When `class_weight='balanced'`, sklearn computes weights as:

$$
w_c = \frac{n_{samples}}{n_{classes} \times n_{samples\_in\_class\_c}}
$$

This means:
- Classes with fewer samples get higher weights  
- Classes with more samples get lower weights  

In [ ]:
# Train Logistic Regression with class weights

# class_weight='balanced' tells the model:
# "Pay more attention to the minority class (fraud)"
model_weighted = LogisticRegression(
    max_iter=1000,
    class_weight='balanced' # ← the only change
)

model_weighted.fit(X_train_scaled, y_train)

In [ ]:
# Predict using the new model

y_pred_w = model_weighted.predict(X_test_scaled)

In [ ]:
# Evaluate the new model

print(classification_report(y_test, y_pred_w))

## Key Observations

- Recall for fraud improved significantly (0.63 → 0.92)  
- Precision for fraud dropped (0.83 → 0.06)  
- Accuracy decreased slightly  

The model now detects more fraud but at the cost of more false positives  

This highlights a key trade-off in imbalanced classification problems.

In [ ]:
# Confusion matrix for weighted model

# Generate confusion matrix
cm_w = confusion_matrix(y_test, y_pred_w)

# Visualize with clear labels
sns.heatmap(
    cm_w,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=['Legitimate', 'Fraud'],   # Predicted labels
    yticklabels=['Legitimate', 'Fraud']    # Actual labels
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Weighted Model)")
plt.show()

Notice how the bottom-left (FN) has reduced — we are missing fewer fraud cases.

## 8. Other Imbalance Techniques (Further Reading)

Today we focused on **algorithm-level handling** using class weights — this is usually the best place to start.

There are other approaches you should be aware of:

| Technique | What it does | When to use |
|-----------|-------------|-------------|
| **SMOTE** | Creates synthetic fraud samples | When class weights alone aren’t enough |
| **Random Undersampling** | Removes majority class samples | When dataset is very large |
| **SMOTE + Undersampling** | Combines both approaches | Common practical setup |
| **Threshold tuning** | Adjusts decision threshold (not just 0.5) | When balancing precision vs recall |
| **Ensemble methods** | e.g. BalancedRandomForest, EasyEnsemble | When using tree-based models |

💡 In practice, you’ll often combine these techniques depending on the problem.

---

### Important Rule

👉 Always apply resampling **only on the training set**

Never resample validation or test data — this gives an unrealistic view of performance.

---

### Further Reading
- https://imbalanced-learn.org/stable/
- https://arxiv.org/abs/1106.1813
- https://developers.google.com/machine-learning/data-prep/construct/sampling-splitting/imbalanced-data

## 9. Saving the Model

We will save our trained model using `joblib`. In **Session 2**, we will load this file and wrap it in a FastAPI endpoint to serve real-time predictions.

In [ ]:
# Note that Colab doesn’t persist files, so we'll have to save our model
# to Google Drive to reuse it

# First we mount google drive (Like bring it here so our notebook can see it)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Save the model and scaler in to a specific folder in your drive.
# My folder is "pydata-fraud-session"

# Define folder path
path = "/content/drive/MyDrive/pydata-fraud-session/"

# Save files
joblib.dump(model_weighted, path + "fraud_model.pkl")
joblib.dump(scaler, path + "scaler.pkl")

In [ ]:
X_test.iloc[[0]]

In [ ]:
# Verify: reload model and scaler, then make a prediction

# Load both model and scaler from the saved folder
loaded_model  = joblib.load(path + "fraud_model.pkl")
loaded_scaler = joblib.load(path + "scaler.pkl")

# Take one sample from the test set
sample = X_test.iloc[[0]]

# Scale with the loaded scaler
sample_scaled = loaded_scaler.transform(sample)

# Make prediction
pred = loaded_model.predict(sample_scaled)[0]
prob = loaded_model.predict_proba(sample_scaled)[0][1]

# Display results
print(f"Sample prediction: {'Fraud' if pred == 1 else 'Legitimate'}")
print(f"Fraud probability: {prob:.4f}")
print(f"Actual label: {'Fraud' if y_test.iloc[0] == 1 else 'Legitimate'}")

In [ ]:
# Get a fraud sample
fraud_sample = X_test[y_test == 1].iloc[0]

print(fraud_sample)

In [ ]:
fraud_sample

## 10. Key Takeaways

### Concepts
1. Frame the problem first — understand what errors cost before building a model  
2. Accuracy can be misleading on imbalanced data  
3. Recall is critical in fraud detection — missing fraud is costly  
4. Class weights are a powerful first step (`class_weight='balanced'`)  
5. Model performance is about trade-offs, not perfection  
6. Threshold tuning is a business decision, not just a technical one  

---

### Skills Practised
- Exploring an imbalanced dataset  
- Training and comparing Logistic Regression models  
- Interpreting confusion matrices and classification reports  
- Improving model performance using class weights  
- Saving a model for deployment  

---

## Session 2 Preview: Deployment with FastAPI

In the next session, we will:
- Load the saved model  
- Build a FastAPI app  
- Serve real-time fraud predictions  

> You can read about APIs here:
https://fastapi.tiangolo.com/

## Bonus Exercise

Try replacing Logistic Regression with a Random Forest:

```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

# Train and evaluate the model

Questions:

- Does recall improve?
- What happens to precision?